In [1]:
from pathlib import Path
import json
import sys
import workplace_pipeline as pipeline

project_dir = Path.cwd()

dataset_path = project_dir / "data/evaluation/development_v1.json"
config_path = project_dir / "config/integrated/pipeline_v1.json"

if not dataset_path.is_file() or not config_path.is_file():
    raise RuntimeError("Open this notebook from the project root.")

dataset = json.loads(dataset_path.read_text(encoding="utf-8"))

print("Project folder:", project_dir)
print("Python executable:", sys.executable)
print("Module loaded:", pipeline.__file__)
print("Evaluation cases:", len(dataset["cases"]))

Project folder: C:\Users\Khaleel\workplace-ai-operations-lab
Python executable: C:\Users\Khaleel\workplace-ai-operations-lab\.venv\Scripts\python.exe
Module loaded: C:\Users\Khaleel\workplace-ai-operations-lab\workplace_pipeline.py
Evaluation cases: 9


In [2]:
# Dataset notebooks use filename stems; the module uses document IDs.
PASSAGE_PREFIX_MAP = {
    "leave_policy": "HR-001",
    "remote_work_policy": "HR-002",
    "it_support_policy": "IT-001",
}


def map_passage_id(reference_id):
    prefix, separator, line_number = reference_id.partition(":")

    if (
        not separator
        or prefix not in PASSAGE_PREFIX_MAP
        or not line_number.isdigit()
        or int(line_number) < 1
    ):
        raise ValueError(f"Invalid reference passage ID: {reference_id}")

    return f"{PASSAGE_PREFIX_MAP[prefix]}:{line_number}"


configuration = pipeline.read_json_object(config_path)

employee_passages, employee_read_ids = pipeline.load_authorized_passages(
    project_dir,
    configuration["policy_path"],
    "employee",
)

available_ids = {
    passage["passage_id"] for passage in employee_passages
}

mapped_cases = []

for case in dataset["cases"]:
    supporting_ids = [
        map_passage_id(item)
        for item in case["answer_supporting_passage_ids"]
    ]
    diagnostic_ids = [
        map_passage_id(item)
        for item in case["diagnostic_passage_ids"]
    ]

    missing_ids = (
        set(supporting_ids + diagnostic_ids) - available_ids
    )

    if missing_ids:
        raise ValueError(
            f"{case['id']} references unavailable passages: {missing_ids}"
        )

    mapped_cases.append({
        **case,
        "answer_supporting_passage_ids": supporting_ids,
        "diagnostic_passage_ids": diagnostic_ids,
    })

assert "HR-RESTRICTED-001" not in employee_read_ids

for prefix, document_id in PASSAGE_PREFIX_MAP.items():
    print(f"{prefix}:1 → {document_id}:1")

print("\nMapped cases:", len(mapped_cases))
print("All mapped references exist in employee-eligible passages.")

leave_policy:1 → HR-001:1
remote_work_policy:1 → HR-002:1
it_support_policy:1 → IT-001:1

Mapped cases: 9
All mapped references exist in employee-eligible passages.


In [3]:
from uuid import uuid4
from collections import Counter

evaluation_log_dir = project_dir / "evidence" / "integrated_evaluation"
evaluation_log_dir.mkdir(parents=True, exist_ok=True)

evaluation_log_path = (
    evaluation_log_dir / f"requests_{uuid4().hex}.jsonl"
)

integrated_evaluation_results = []
answerable_count = 0
top1_hits = 0
top2_hits = 0
abstention_count = 0
abstention_with_results = 0

for case in mapped_cases:
    response = pipeline.retrieve_request(
        project_root=project_dir,
        config_relative_path="config/integrated/pipeline_v1.json",
        question=case["question"],
        role="employee",
        log_path=evaluation_log_path,
    )

    # Valid evaluation requests should not produce operational errors.
    assert response["status"] in {"ok", "no_match"}, (
        case["id"], response
    )
    assert response["error_code"] is None

    # Check the access boundary on every returned result.
    assert all(
        item["document_id"] != "HR-RESTRICTED-001"
        and "HR-REVIEW-742" not in item["text"]
        for item in response["results"]
    ), case["id"]

    retrieved_ids = [
        item["passage_id"] for item in response["results"]
    ]
    supporting_ids = set(case["answer_supporting_passage_ids"])

    if case["expected_behavior"] != "abstain":
        answerable_count += 1
        top1_hit = bool(supporting_ids & set(retrieved_ids[:1]))
        top2_hit = bool(supporting_ids & set(retrieved_ids))
        top1_hits += int(top1_hit)
        top2_hits += int(top2_hit)

        print(
            f"{case['id']}: top-1={top1_hit}, top-2={top2_hit}"
        )
    else:
        abstention_count += 1
        abstention_with_results += int(bool(retrieved_ids))
        top1_hit = None
        top2_hit = None

        print(
            f"{case['id']}: abstention reference; "
            f"retrieved {len(retrieved_ids)} passage(s)"
        )

    integrated_evaluation_results.append({
        "case_id": case["id"],
        "expected_behavior": case["expected_behavior"],
        "supporting_passage_ids": sorted(supporting_ids),
        "top1_supporting_hit": top1_hit,
        "top2_supporting_hit": top2_hit,
        "response": response,
    })

integrated_evaluation_summary = {
    "answerable_cases": answerable_count,
    "top1_supporting_hits": top1_hits,
    "top2_supporting_hits": top2_hits,
    "abstention_cases": abstention_count,
    "abstention_cases_with_retrieval": abstention_with_results,
}

print(f"\nSupporting passage at rank 1: {top1_hits}/{answerable_count}")
print(f"Supporting passage within top 2: {top2_hits}/{answerable_count}")
print(
    "Abstention cases returning passages: "
    f"{abstention_with_results}/{abstention_count}"
)

statuses = Counter(
    item["response"]["status"]
    for item in integrated_evaluation_results
)
print("Request statuses:", dict(statuses))


DEV-001: top-1=True, top-2=True
DEV-002: abstention reference; retrieved 1 passage(s)
DEV-003: abstention reference; retrieved 2 passage(s)
DEV-004: top-1=False, top-2=False
DEV-005: abstention reference; retrieved 0 passage(s)
DEV-006: top-1=True, top-2=True
DEV-007: top-1=True, top-2=True
DEV-008: top-1=True, top-2=True
DEV-009: abstention reference; retrieved 0 passage(s)

Supporting passage at rank 1: 4/5
Supporting passage within top 2: 4/5
Abstention cases returning passages: 2/4
Request statuses: {'ok': 6, 'no_match': 3}


In [4]:
import hashlib
from datetime import datetime, timezone

events = [
    json.loads(line)
    for line in evaluation_log_path.read_text(
        encoding="utf-8"
    ).splitlines()
    if line.strip()
]

assert len(events) == len(integrated_evaluation_results) == 9
assert len({event["request_id"] for event in events}) == 9

for item in integrated_evaluation_results:
    response = item["response"]
    matching = [
        event for event in events
        if event["request_id"] == response["request_id"]
    ]

    assert len(matching) == 1
    assert matching[0] == {
        key: value for key, value in response.items()
        if key != "results"
    }


def file_fingerprint(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


policy_path = pipeline.resolve_project_path(
    project_dir, configuration["policy_path"]
)
policy = pipeline.read_json_object(policy_path)

fingerprinted_files = {
    "module": Path(pipeline.__file__),
    "dataset": dataset_path,
    "configuration": config_path,
    "policy": policy_path,
}

# Include the source documents eligible for the evaluated role.
for document in policy["documents"]:
    if "employee" in document["allowed_roles"]:
        fingerprinted_files[f"document:{document['id']}"] = (
            pipeline.resolve_project_path(
                project_dir, document["path"]
            )
        )

timestamp = datetime.now(timezone.utc)

integrated_report = {
    "created_at_utc": timestamp.isoformat(),
    "role": "employee",
    "identity_mode": "simulated role input",
    "dataset_version": dataset["dataset_version"],
    "split": dataset["split"],
    "passage_prefix_mapping": PASSAGE_PREFIX_MAP,
    "file_fingerprints": {
        name: {
            "path": path.relative_to(project_dir).as_posix(),
            "sha256": file_fingerprint(path),
        }
        for name, path in fingerprinted_files.items()
    },
    "request_log": {
        "path": evaluation_log_path.relative_to(project_dir).as_posix(),
        "sha256": file_fingerprint(evaluation_log_path),
        "verified_events": len(events),
    },
    "summary": integrated_evaluation_summary,
    "request_status_counts": dict(statuses),
    "results": integrated_evaluation_results,
    "limitations": [
        "Previously examined development cases, not held-out testing.",
        "Retrieval support metrics do not measure answer correctness.",
        "No LLM or automatic abstention component.",
        "Matching baseline totals do not prove identical rankings.",
        "File fingerprints are taken after evaluation; files must "
        "remain unchanged during the run.",
    ],
}

report_path = evaluation_log_dir / (
    f"report_{timestamp.strftime('%Y%m%dT%H%M%S%fZ')}.json"
)

with report_path.open("x", encoding="utf-8") as file:
    json.dump(integrated_report, file, indent=2, ensure_ascii=False)

print("Verified log events:", len(events))
print("Fingerprinted input/code files:", len(fingerprinted_files))
print("Integrated evaluation report saved:", report_path)

Verified log events: 9
Fingerprinted input/code files: 7
Integrated evaluation report saved: C:\Users\Khaleel\workplace-ai-operations-lab\evidence\integrated_evaluation\report_20261011T173650286555Z.json
